<a href="https://colab.research.google.com/github/uniesecruz/ESALQ_money_laundering/blob/HI-medium/TCC_ESALQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Instalação das bibliotecas

In [1]:
# Instalar pyspark
!pip install pyspark

# Leitura do arquivos CSV com spark

In [2]:
from pyspark.sql import SparkSession

# Ajustado para os 167GB reais da sua instância
memory_limit = "140g"

spark = SparkSession.builder \
    .appName("TCC_AML_HighPerformance") \
    .config("spark.driver.memory", memory_limit) \
    .config("spark.executor.memory", memory_limit) \
    .config("spark.driver.maxResultSize", "30g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.3") \
    .getOrCreate()

print(f"Spark inicializado com {memory_limit} de RAM disponível.")

Spark inicializado com 140g de RAM disponível.


In [ ]:
# # Carregar o arquivo CSV usando Spark
# spark_trans_df = spark.read.csv('/content/drive/MyDrive/TCC/data/external/HI-Medium_Trans.csv', header=True, inferSchema=True)

# print("Schema do Spark DataFrame de Transações:")
# spark_trans_df.printSchema()

# print("Primeiras 5 linhas do Spark DataFrame de Transações:")
# spark_trans_df.show(5)

Schema do Spark DataFrame de Transações:
root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)

Primeiras 5 linhas do Spark DataFrame de Transações:
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|       Timestamp|From Bank| Account2|To Bank| Account4|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+--------------

In [ ]:
# # Carregar o arquivo CSV de contas usando Spark
# spark_accounts_df = spark.read.csv('/content/drive/MyDrive/TCC/data/external/HI-Medium_accounts.csv', header=True, inferSchema=True)

# print("Schema do Spark DataFrame de Contas:")
# spark_accounts_df.printSchema()

# print("Primeiras 5 linhas do Spark DataFrame de Contas:")
# spark_accounts_df.show(5)

Schema do Spark DataFrame de Contas:
root
 |-- Bank Name: string (nullable = true)
 |-- Bank ID: integer (nullable = true)
 |-- Account Number: string (nullable = true)
 |-- Entity ID: string (nullable = true)
 |-- Entity Name: string (nullable = true)

Primeiras 5 linhas do Spark DataFrame de Contas:
+--------------------+-------+--------------+-----------+--------------------+
|           Bank Name|Bank ID|Account Number|  Entity ID|         Entity Name|
+--------------------+-------+--------------+-----------+--------------------+
|     China Bank #561|  53267|     817D00980|2AA1F24F180| Corporation #183669|
|   Spain Bank #18657| 316997|     808BB2280|2AA1EEB8540| Partnership #193780|
|First Bank of Helena| 339367|     8505ED380|2AA206D7790|Sole Proprietorsh...|
|   Mexico Bank #3367|3148419|     8363D4180|2AA2001B1A0| Partnership #133577|
|Switzerland Bank ...|3174937|     842090C80|2AA20224CB0|Sole Proprietorsh...|
+--------------------+-------+--------------+-----------+--------

Salvando os arquivos em parquet

In [ ]:
# spark_trans_df.write.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_Trans', mode='overwrite')
# print("spark_trans_df salvo em parquet.")

# spark_accounts_df.write.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_accounts', mode='overwrite')
# print("spark_accounts_df salvo em parquet.")

spark_trans_df salvo em parquet.
spark_accounts_df salvo em parquet.


# Lendo o arquivo parquet (Começar aqui)

In [ ]:
# from pyspark.sql import SparkSession
# import os

# # 1. Definir a memória máxima para o driver (essencial no Colab)
# # Deixamos uma margem de segurança para o Sistema Operacional (~4-5GB)
# memory_limit = "46g"

# spark = SparkSession.builder \
#     .appName("Max_Performance_Spark") \
#     .config("spark.driver.memory", memory_limit) \
#     .config("spark.executor.memory", memory_limit) \
#     .config("spark.driver.maxResultSize", "10g") \
#     .config("spark.sql.shuffle.partitions", "200") \
#     .config("spark.memory.fraction", "0.8") \
#     .config("spark.memory.storageFraction", "0.3") \
#     .config("spark.ui.port", "4050") \
#     .getOrCreate()

# print(f"SparkSession inicializada com foco em High-RAM ({memory_limit}).")

SparkSession inicializada com foco em High-RAM (46g).


In [ ]:
spark_trans_df= spark.read.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_Trans')
spark_accounts_df = spark.read.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_accounts')

# Profiling Trans

In [ ]:
print("### Profiling do spark_trans_df ###")

print("\nSchema do DataFrame:")
spark_trans_df.printSchema()

print("\nTipos de Dados das Colunas:")
for col, dtype in spark_trans_df.dtypes:
    print(f"{col}: {dtype}")

print("\nEstatísticas Descritivas (describe()):")
spark_trans_df.describe().show()

print("\nEstatísticas Sumárias (summary()):")
spark_trans_df.summary().show()

print(f"\nNúmero total de linhas: {spark_trans_df.count()}")

print("\nNomes das Colunas:")
print(spark_trans_df.columns)

### Profiling do spark_trans_df ###

Schema do DataFrame:
root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)


Tipos de Dados das Colunas:
Timestamp: string
From Bank: int
Account2: string
To Bank: int
Account4: string
Amount Received: double
Receiving Currency: string
Amount Paid: double
Payment Currency: string
Payment Format: string
Is Laundering: int

Estatísticas Descritivas (describe()):
+-------+----------------+-----------------+---------+------------------+---------+--------------------+------------------+--------------------+-----------------+----

# Profiling Accounts

In [ ]:
print("### Profiling do spark_accounts_df ###")

print("\nSchema do DataFrame:")
spark_accounts_df.printSchema()

print("\nTipos de Dados das Colunas:")
for col, dtype in spark_accounts_df.dtypes:
    print(f"{col}: {dtype}")

print("\nEstatísticas Descritivas (describe()):")
spark_accounts_df.describe().show()

print("\nEstatísticas Sumárias (summary()):")
spark_accounts_df.summary().show()

print(f"\nNúmero total de linhas: {spark_accounts_df.count()}")

print("\nNomes das Colunas:")
print(spark_accounts_df.columns)

### Profiling do spark_accounts_df ###

Schema do DataFrame:
root
 |-- Bank Name: string (nullable = true)
 |-- Bank ID: integer (nullable = true)
 |-- Account Number: string (nullable = true)
 |-- Entity ID: string (nullable = true)
 |-- Entity Name: string (nullable = true)


Tipos de Dados das Colunas:
Bank Name: string
Bank ID: int
Account Number: string
Entity ID: string
Entity Name: string

Estatísticas Descritivas (describe()):
+-------+------------------+-----------------+--------------+-----------+--------------------+
|summary|         Bank Name|          Bank ID|Account Number|  Entity ID|         Entity Name|
+-------+------------------+-----------------+--------------+-----------+--------------------+
|  count|           2087786|          2087786|       2087786|    2087786|             2087786|
|   mean|              NULL|632102.3694339362|      Infinity|       NULL|                NULL|
| stddev|              NULL|970374.5529627781|           NaN|       NULL|             

In [ ]:
spark_trans_df =spark_trans_df.withColumnRenamed("Account2", "From Account") \
                                     .withColumnRenamed("Account4", "To Account")

# Join dos datasets

In [ ]:

from pyspark.sql import functions as F



# 2. Renomear colunas de spark_trans_df (Equivalente ao spark_trans_df.columns no Pandas)
# Nota: Como o Spark não aceita atribuição direta de lista em .columns, usamos select e alias
new_columns = ['Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account',
               'Amount Received', 'Receiving Currency', 'Amount Paid',
               'Payment Currency', 'Payment Format', 'Is Laundering']

spark_trans_df = spark_trans_df.toDF(*new_columns)

# 3. Preparar spark_accounts_df para os joins (selecionando apenas colunas necessárias)
# Isso evita colisões de nomes e facilita o mapeamento posterior
acc_info = spark_accounts_df.select(
    F.col("Bank ID"),
    F.col("Account Number"),
    F.col("Bank Name"),
    F.col("Entity ID"),
    F.col("Entity Name")
)

# --- JOIN 1: Informações da conta de ORIGEM (Sender) ---
trans_enriched_df = spark_trans_df.join(
    F.broadcast(acc_info),
    (spark_trans_df["From Bank"] == acc_info["Bank ID"]) &
    (spark_trans_df["From Account"] == acc_info["Account Number"]),
    how='left'
).select(
    spark_trans_df["*"], # Mantém todas as colunas originais da transação
    F.col("Bank Name").alias("From Bank Name"),
    F.col("Entity ID").alias("From Entity ID"),
    F.col("Entity Name").alias("From Entity Name")
)

# --- JOIN 2: Informações da conta de DESTINO (Receiver) ---
trans_enriched_df = trans_enriched_df.join(
    F.broadcast(acc_info),
    (trans_enriched_df["To Bank"] == acc_info["Bank ID"]) &
    (trans_enriched_df["To Account"] == acc_info["Account Number"]),
    how='left'
).select(
    trans_enriched_df["*"], # Mantém as colunas do primeiro join
    F.col("Bank Name").alias("To Bank Name"),
    F.col("Entity ID").alias("To Entity ID"),
    F.col("Entity Name").alias("To Entity Name")
)

# Exibir o resultado
print("Tabela de Transações Enriquecida:")
trans_enriched_df.show(5)

# Se precisar contar o total (equivalente ao print final)
# print(f"Total de registros: {trans_enriched_df.count()}")

Tabela de Transações Enriquecida:
+----------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+--------------------+--------------+--------------------+--------------------+------------+------------------+
|       Timestamp|From Bank|From Account|To Bank|To Account|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|      From Bank Name|From Entity ID|    From Entity Name|        To Bank Name|To Entity ID|    To Entity Name|
+----------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+--------------------+--------------+--------------------+--------------------+------------+------------------+
|2022/09/01 11:02|   207628|   84CD42740| 207628| 84CD42740|           3.46|         US Dollar|       3.46|       US Dollar|  Reinvestment|            0|Willows Savings Bank|

# Salvando base enriquecida em parquet

In [ ]:
# trans_enriched_df.write.parquet('/content/drive/MyDrive/TCC/data/processed/parquet/HI-Medium_enriched', mode='overwrite')
# print("trans_enriched_df salvo em parquet.")

trans_enriched_df salvo em parquet.


Realizando Leitura da base enriquecida em parquet

In [ ]:
trans_enriched_df = spark.read.parquet('/content/drive/MyDrive/TCC/data/processed/parquet/HI-Medium_enriched')

In [ ]:
from dataclasses import dataclass, field
from typing import Dict
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql import functions as F

SECONDS_BY_WINDOW = {
    "1h": 3600,
    "24h": 86400,
    "7d": 604800,
    "30d": 2592000,
}

@dataclass
class FeatureEngineeringPipelineSpark:
    """Generate AML features using deterministic Spark window operations."""

    timestamp_col: str = "Timestamp"
    account_col: str = "From Account"
    amount_col: str = "Amount Received"
    bank_col: str = "Receiving Currency"
    country_col: str = "From Bank"
    velocity_windows: Dict[str, str] = field(
        default_factory=lambda: {"1h": "1h", "24h": "24h", "7d": "7d"}
    )
    ratio_window: str = "30d"
    smurf_threshold: float = 10000.0

    def fit(self, df: DataFrame) -> "FeatureEngineeringPipelineSpark":
        """No learned state; kept for API compatibility."""
        _ = df
        return self

    def transform(self, df: DataFrame) -> DataFrame:
        """Apply full feature engineering pipeline in Spark."""
        base = (
            df.withColumn(self.timestamp_col, F.to_timestamp(F.col(self.timestamp_col), 'yyyy/MM/dd HH:mm'))
            .filter(F.col(self.timestamp_col).isNotNull())
            .withColumn("_ts_long", F.col(self.timestamp_col).cast("long"))
        )

        # Deterministic ordering for lag-based features.
        order_cols = [
            F.col(self.timestamp_col).asc(),
            F.col(self.amount_col).asc_nulls_last(),
            F.col("To Account").asc_nulls_last(),
        ]
        row_win = Window.partitionBy(self.account_col).orderBy(*order_cols)

        enriched = self._add_velocity_features(base)
        enriched = self._add_ratio_features(enriched)
        enriched = self._add_behavioral_features(enriched, row_win, order_cols)
        enriched = self._add_smurfing_features(enriched)

        enriched = enriched.drop("_ts_long")
        return enriched

    def fit_transform(self, df: DataFrame) -> DataFrame:
        return self.fit(df).transform(df)

    def _add_velocity_features(self, df: DataFrame) -> DataFrame:
        out = df
        for window_name in self.velocity_windows:
            seconds = SECONDS_BY_WINDOW[window_name]
            win = (
                Window.partitionBy(self.account_col)
                .orderBy(F.col("_ts_long"))
                .rangeBetween(-seconds, -1)
            )
            out = out.withColumn(
                f"txn_count_{window_name}_velocity",
                F.coalesce(F.count(F.lit(1)).over(win), F.lit(0)).cast("double"),
            )
            out = out.withColumn(
                f"amount_sum_{window_name}_velocity",
                F.coalesce(F.sum(F.col(self.amount_col)).over(win), F.lit(0.0)),
            )
            out = out.withColumn(
                f"amount_mean_{window_name}_velocity",
                F.coalesce(F.avg(F.col(self.amount_col)).over(win), F.lit(0.0)),
            )
            out = out.withColumn(
                f"amount_max_{window_name}_velocity",
                F.coalesce(F.max(F.col(self.amount_col)).over(win), F.lit(0.0)),
            )
            if window_name == "7d":
                out = out.withColumn(
                    f"amount_std_{window_name}_velocity",
                    F.coalesce(F.stddev(F.col(self.amount_col)).over(win), F.lit(0.0)),
                )

        return out

    def _add_ratio_features(self, df: DataFrame) -> DataFrame:
        seconds = SECONDS_BY_WINDOW[self.ratio_window]
        win = (
            Window.partitionBy(self.account_col)
            .orderBy(F.col("_ts_long"))
            .rangeBetween(-seconds, -1)
        )

        hist_mean = F.avg(F.col(self.amount_col)).over(win)
        hist_max = F.max(F.col(self.amount_col)).over(win)
        hist_std = F.stddev(F.col(self.amount_col)).over(win)

        out = df.withColumn("_hist_mean", hist_mean)
        out = out.withColumn("_hist_max", hist_max)
        out = out.withColumn("_hist_std", hist_std)

        out = out.withColumn(
            "amount_to_historical_mean_ratio",
            F.when(F.col("_hist_mean") > 0, F.col(self.amount_col) / F.col("_hist_mean")).otherwise(0.0),
        )
        out = out.withColumn(
            "amount_to_historical_max_ratio",
            F.when(F.col("_hist_max") > 0, F.col(self.amount_col) / F.col("_hist_max")).otherwise(0.0),
        )
        out = out.withColumn(
            "amount_zscore_historical",
            F.when(F.col("_hist_std") > 0, (F.col(self.amount_col) - F.col("_hist_mean")) / F.col("_hist_std")).otherwise(0.0),
        )

        return out.drop("_hist_mean", "_hist_max", "_hist_std")

    def _add_behavioral_features(
        self,
        df: DataFrame,
        row_win: Window,
        order_cols: list,
    ) -> DataFrame:
        out = df

        lag_ts = F.lag(F.col("_ts_long")).over(row_win)
        out = out.withColumn(
            "time_since_last_txn_seconds",
            F.coalesce((F.col("_ts_long") - lag_ts).cast("double"), F.lit(0.0)),
        )

        lag_bank = F.lag(F.col(self.bank_col)).over(row_win)
        out = out.withColumn(
            "bank_change_flag",
            F.when(lag_bank.isNull(), F.lit(0)).when(F.col(self.bank_col) != lag_bank, F.lit(1)).otherwise(F.lit(0)),
        )

        # 1 if this is the first time account->country pair appears.
        country_win = Window.partitionBy(self.account_col, self.country_col).orderBy(*order_cols)
        out = out.withColumn(
            "is_new_country",
            F.when(F.row_number().over(country_win) == 1, F.lit(1)).otherwise(F.lit(0)),
        )

        out = out.withColumn("hour_of_day", F.hour(F.col(self.timestamp_col)))
        out = out.withColumn(
            "is_unusual_hour",
            F.when((F.col("hour_of_day") < 6) | (F.col("hour_of_day") > 22), F.lit(1)).otherwise(F.lit(0)),
        )

        return out

    def _add_smurfing_features(self, df: DataFrame) -> DataFrame:
        win24 = (
            Window.partitionBy(self.account_col)
            .orderBy(F.col("_ts_long"))
            .rangeBetween(-SECONDS_BY_WINDOW["24h"], -1)
        )

        is_smurf = (F.col(self.amount_col) >= self.smurf_threshold * 0.8) & (
            F.col(self.amount_col) < self.smurf_threshold
        )
        smurf_amount = F.when(is_smurf, F.col(self.amount_col)).otherwise(F.lit(0.0))
        proximity = F.when(
            F.col(self.amount_col) < self.smurf_threshold,
            (F.lit(self.smurf_threshold) - F.col(self.amount_col)) / F.lit(self.smurf_threshold),
        ).otherwise(F.lit(0.0))

        out = df.withColumn(
            "smurf_txn_count_24h_behavioral",
            F.coalesce(F.sum(F.when(is_smurf, 1).otherwise(0)).over(win24).cast("double"), F.lit(0.0)),
        )
        out = out.withColumn(
            "smurf_amount_sum_24h_behavioral",
            F.coalesce(F.sum(smurf_amount).over(win24), F.lit(0.0)),
        )
        out = out.withColumn(
            "smurf_proximity_score_behavioral",
            F.coalesce(F.avg(proximity).over(win24), F.lit(0.0)),
        )

        return out

# Instantiate the feature engineering pipeline
feature_pipeline = FeatureEngineeringPipelineSpark(
    timestamp_col="Timestamp",
    account_col="From Account",
    amount_col="Amount Received",
    bank_col="Receiving Currency",
    country_col="From Bank",
    velocity_windows={"1h": "1h", "24h": "24h", "7d": "7d"},
    ratio_window="30d",
    smurf_threshold=10000.0
)

# Apply the transformation to the enriched transactions DataFrame
df_features = feature_pipeline.transform(trans_enriched_df)

# Display the schema and some data with the new features
print("Schema do DataFrame com Features:")
df_features.printSchema()

print("Primeiras 5 linhas do DataFrame com Features:")
df_features.show(5)

Schema do DataFrame com Features:
root
 |-- Timestamp: timestamp (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- From Account: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- To Account: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)
 |-- From Bank Name: string (nullable = true)
 |-- From Entity ID: string (nullable = true)
 |-- From Entity Name: string (nullable = true)
 |-- To Bank Name: string (nullable = true)
 |-- To Entity ID: string (nullable = true)
 |-- To Entity Name: string (nullable = true)
 |-- txn_count_1h_velocity: double (nullable = false)
 |-- amount_sum_1h_velocity: double (nullable = false)
 |-- amount_mean_1h_velocity: double (nullable = false)
 |-- amount_max_1h_velocity: d

In [ ]:
# df_features.write.parquet('/content/drive/MyDrive/TCC/data/features', mode='overwrite')
# print("df_features salvo em parquet.")

In [3]:
df_features = spark.read.parquet('/content/drive/MyDrive/TCC/data/features')

In [4]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

# EDA

### Análise da Variável Alvo: `Is Laundering`

In [5]:
print("Distribuição da variável alvo 'Is Laundering':")
df_features.groupBy("Is Laundering").count().show()

Distribuição da variável alvo 'Is Laundering':
+-------------+--------+
|Is Laundering|   count|
+-------------+--------+
|            0|31863008|
|            1|   35230|
+-------------+--------+



### Análise da `Amount Received` por `Is Laundering`

In [6]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F
print("Estatísticas de 'Amount Received' por 'Is Laundering':")
df_features.groupBy("Is Laundering").agg(
    F.mean("Amount Received").alias("Mean Amount Received"),
    F.stddev("Amount Received").alias("StdDev Amount Received"),
    F.min("Amount Received").alias("Min Amount Received"),
    F.max("Amount Received").alias("Max Amount Received")
).show()

Estatísticas de 'Amount Received' por 'Is Laundering':
+-------------+--------------------+----------------------+-------------------+-------------------+
|Is Laundering|Mean Amount Received|StdDev Amount Received|Min Amount Received|Max Amount Received|
+-------------+--------------------+----------------------+-------------------+-------------------+
|            0|   6379497.069583804|  2.5888672647249527E9|             1.0E-6|8.15860932172761E12|
|            1| 5.311674507372348E7|   4.988977908002805E9|             2.8E-5| 9.0627007807088E11|
+-------------+--------------------+----------------------+-------------------+-------------------+



### Análise das Features de Velocidade (Exemplo: `txn_count_1h_velocity`)

In [7]:
print("Estatísticas de 'txn_count_1h_velocity' por 'Is Laundering':")
df_features.groupBy("Is Laundering").agg(
    F.mean("txn_count_1h_velocity").alias("Mean Txn Count 1h"),
    F.stddev("txn_count_1h_velocity").alias("StdDev Txn Count 1h"),
    F.max("txn_count_1h_velocity").alias("Max Txn Count 1h")
).show()

Estatísticas de 'txn_count_1h_velocity' por 'Is Laundering':
+-------------+------------------+-------------------+----------------+
|Is Laundering| Mean Txn Count 1h|StdDev Txn Count 1h|Max Txn Count 1h|
+-------------+------------------+-------------------+----------------+
|            0| 162.1870955497987|  662.7356778169013|         10909.0|
|            1|206.80706783990917|  741.6951238929706|         10193.0|
+-------------+------------------+-------------------+----------------+



### Análise das Features de Smurfing (Exemplo: `smurf_txn_count_24h_behavioral`)

In [8]:
print("Estatísticas de 'smurf_txn_count_24h_behavioral' por 'Is Laundering':")
df_features.groupBy("Is Laundering").agg(
    F.mean("smurf_txn_count_24h_behavioral").alias("Mean Smurf Txn Count 24h"),
    F.stddev("smurf_txn_count_24h_behavioral").alias("StdDev Smurf Txn Count 24h"),
    F.max("smurf_txn_count_24h_behavioral").alias("Max Smurf Txn Count 24h")
).show()

Estatísticas de 'smurf_txn_count_24h_behavioral' por 'Is Laundering':
+-------------+------------------------+--------------------------+-----------------------+
|Is Laundering|Mean Smurf Txn Count 24h|StdDev Smurf Txn Count 24h|Max Smurf Txn Count 24h|
+-------------+------------------------+--------------------------+-----------------------+
|            0|       72.68752460533544|         296.4559441726601|                 2590.0|
|            1|       92.97715015611695|         330.7150875290676|                 2578.0|
+-------------+------------------------+--------------------------+-----------------------+



In [9]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F


# Order the DataFrame by Timestamp to ensure temporal split
df_features_sorted = df_features.orderBy(col('Timestamp'))

# Calculate the total number of rows
total_rows = df_features_sorted.count()

# Determine the split point (80% for training, 20% for OOT)
split_index = int(total_rows * 0.8)

# Get the timestamp at the 80% mark. We'll use this as our split date.
split_timestamp = df_features_sorted.select('Timestamp').limit(split_index).collect()[-1][0]

# Split the DataFrame into training and OOT sets
df_train = df_features_sorted.filter(col('Timestamp') <= split_timestamp)
df_oot = df_features_sorted.filter(col('Timestamp') > split_timestamp)

print(f"Total de registros: {total_rows}")
print(f"Timestamp para o corte (80% dos dados): {split_timestamp}")
print(f"Registros no conjunto de treino (80%): {df_train.count()}")
print(f"Registros no conjunto OOT (20%): {df_oot.count()}")

print("\nIntervalo de datas do conjunto de treino:")
df_train.agg(F.min('Timestamp').alias('Min Timestamp'), F.max('Timestamp').alias('Max Timestamp')).show()

print("\nIntervalo de datas do conjunto OOT:")
df_oot.agg(F.min('Timestamp').alias('Min Timestamp'), F.max('Timestamp').alias('Max Timestamp')).show()

Total de registros: 31898238
Timestamp para o corte (80% dos dados): 2022-09-14 05:46:00
Registros no conjunto de treino (80%): 25519347
Registros no conjunto OOT (20%): 6378891

Intervalo de datas do conjunto de treino:
+-------------------+-------------------+
|      Min Timestamp|      Max Timestamp|
+-------------------+-------------------+
|2022-09-01 00:00:00|2022-09-14 05:46:00|
+-------------------+-------------------+


Intervalo de datas do conjunto OOT:
+-------------------+-------------------+
|      Min Timestamp|      Max Timestamp|
+-------------------+-------------------+
|2022-09-14 05:47:00|2022-09-28 15:58:00|
+-------------------+-------------------+



In [10]:
# from pyspark.sql import SparkSession
# import os

# # 1. Definir a memória máxima para o driver (essencial no Colab)
# # Deixamos uma margem de segurança para o Sistema Operacional (~4-5GB)
# memory_limit = "46g"

# spark = SparkSession.builder \
#     .appName("Max_Performance_Spark") \
#     .config("spark.driver.memory", memory_limit) \
#     .config("spark.executor.memory", memory_limit) \
#     .config("spark.driver.maxResultSize", "10g") \
#     .config("spark.sql.shuffle.partitions", "200") \
#     .config("spark.memory.fraction", "0.8") \
#     .config("spark.memory.storageFraction", "0.3") \
#     .config("spark.ui.port", "4050") \
#     .getOrCreate()

# print(f"SparkSession inicializada com foco em High-RAM ({memory_limit}).")

In [11]:
# df_train.write.parquet('/content/drive/MyDrive/TCC/data/treino_e_oot/df_train', mode='overwrite')
# print("df_train salvo em parquet.")
df_oot.write.parquet('/content/drive/MyDrive/TCC/data/treino_e_oot/df_oot', mode='overwrite')
print("df_oot salvo em parquet.")

df_oot salvo em parquet.


In [12]:
df_train = spark.read.parquet('/content/drive/MyDrive/TCC/data/treino_e_oot/df_train')
df_oot = spark.read.parquet('/content/drive/MyDrive/TCC/data/treino_e_oot/df_oot')


In [13]:
from pyspark.sql import functions as F

# Definir target
target_col = 'Is Laundering'

# Colunas para remover (além do target)
cols_to_remove = [
    target_col,
    'Timestamp',  # Já temos safra e componentes temporais
    'From Bank', 'To Bank',  # Alta cardinalidade, usar apenas se necessário
    'From Account', 'To Account',  # IDs individuais
    'From Entity ID', 'To Entity ID',  # IDs de entidade
]

# Verificar quais colunas existem antes de remover
cols_to_remove = [col for col in cols_to_remove if col in df_train.columns] # Usar df_train.columns aqui

# Separar X e y
X_train = df_train.drop(*cols_to_remove)
y_train = df_train.select(target_col)

X_oot = df_oot.drop(*cols_to_remove)
y_oot = df_oot.select(target_col)

print("="*80)
print("SEPARAÇÃO DE FEATURES E TARGET")
print("="*80)

print(f"\n📊 Colunas removidas ({len(cols_to_remove)}):")
for col in cols_to_remove:
    print(f"   - {col}")

print(f"\n📊 Treino:")
print(f"   X_train count: {X_train.count()} rows, {len(X_train.columns)} columns") # Spark count and len(columns)
print(f"   y_train count: {y_train.count()} rows, {len(y_train.columns)} columns") # Spark count and len(columns)
# Calculate mean for Spark DataFrame
y_train_mean = y_train.select(F.mean(target_col)).collect()[0][0]
print(f"   Taxa de lavagem: {y_train_mean*100:.2f}%")

print(f"\n📊 OOT:")
print(f"   X_oot count: {X_oot.count()} rows, {len(X_oot.columns)} columns") # Spark count and len(columns)
print(f"   y_oot count: {y_oot.count()} rows, {len(y_oot.columns)} columns") # Spark count and len(columns)
# Calculate mean for Spark DataFrame
y_oot_mean = y_oot.select(F.mean(target_col)).collect()[0][0]
print(f"   Taxa de lavagem: {y_oot_mean*100:.2f}%")

print(f"\n📋 Features disponíveis ({len(X_train.columns)}):")
print(f"   Principais colunas:")
for col in X_train.columns[:15]:
    print(f"   - {col}")
if len(X_train.columns) > 15:
    print(f"   ... e mais {len(X_train.columns) - 15} colunas")

SEPARAÇÃO DE FEATURES E TARGET

📊 Colunas removidas (8):
   - Is Laundering
   - Timestamp
   - From Bank
   - To Bank
   - From Account
   - To Account
   - From Entity ID
   - To Entity ID

📊 Treino:
   X_train count: 25519347 rows, 33 columns
   y_train count: 25519347 rows, 1 columns
   Taxa de lavagem: 0.10%

📊 OOT:
   X_oot count: 6378891 rows, 33 columns
   y_oot count: 6378891 rows, 1 columns
   Taxa de lavagem: 0.17%

📋 Features disponíveis (33):
   Principais colunas:
   - Amount Received
   - Receiving Currency
   - Amount Paid
   - Payment Currency
   - Payment Format
   - From Bank Name
   - From Entity Name
   - To Bank Name
   - To Entity Name
   - txn_count_1h_velocity
   - amount_sum_1h_velocity
   - amount_mean_1h_velocity
   - amount_max_1h_velocity
   - txn_count_24h_velocity
   - amount_sum_24h_velocity
   ... e mais 18 colunas


In [22]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, LongType, DoubleType, FloatType

# Identificar colunas categóricas e numéricas usando o schema do Spark
categorical_cols = [f.name for f in X_train.schema.fields if isinstance(f.dataType, StringType)]
numeric_cols = [f.name for f in X_train.schema.fields if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType))]

# Remover safra se estiver nos numéricos (é identificador, não feature)
# No contexto Spark, 'safra' pode não ser um tipo de dado numérico explícito ou pode não existir.
# Apenas para compatibilidade com o código original, mantemos a lógica.
# if 'safra' in numeric_cols:
#     numeric_cols.remove('safra')

print("\n" + "="*80)
print("IDENTIFICAÇÃO DE COLUNAS")
print("="*80)

print(f"\n📊 Colunas categóricas ({len(categorical_cols)}):")
low_cardinality = []
high_cardinality = []

for col_name in categorical_cols:
    # Para Spark, contar valores únicos pode ser custoso. Usaremos uma estimativa ou contagem exata.
    # Para este exemplo, faremos a contagem exata para replicar a lógica do Pandas.
    nunique = X_train.select(F.col(col_name)).distinct().count()
    print(f"   - {col_name}: {nunique} categorias")

    if nunique <= 20:  # Limite conservador para OneHot
        low_cardinality.append(col_name)
    else:
        high_cardinality.append(col_name)


print(f"\n📊 Categóricas de baixa cardinalidade (≤20): {len(low_cardinality)}")
for col_name in low_cardinality:
    nunique = X_train.select(F.col(col_name)).distinct().count()
    print(f"   - {col_name}: {nunique} categorias")

print(f"\n📊 Categóricas de alta cardinalidade (>20): {len(high_cardinality)}")
for col_name in high_cardinality:
    nunique = X_train.select(F.col(col_name)).distinct().count()
    print(f"   - {col_name}: {nunique} categorias → REMOVIDA")

# Ajustar categóricas para usar apenas baixa cardinalidade
categorical_cols = low_cardinality

print(f"\n📊 Colunas numéricas ({len(numeric_cols)}):")
for col_name in numeric_cols[:10]:
    print(f"   - {col_name}")
if len(numeric_cols) > 10:
    print(f"   ... e mais {len(numeric_cols) - 10} colunas")


IDENTIFICAÇÃO DE COLUNAS

📊 Colunas categóricas (7):
   - Receiving Currency: 15 categorias
   - Payment Currency: 15 categorias
   - Payment Format: 7 categorias
   - From Bank Name: 80178 categorias
   - From Entity Name: 641590 categorias
   - To Bank Name: 41882 categorias
   - To Entity Name: 664021 categorias

📊 Categóricas de baixa cardinalidade (≤20): 3
   - Receiving Currency: 15 categorias
   - Payment Currency: 15 categorias
   - Payment Format: 7 categorias

📊 Categóricas de alta cardinalidade (>20): 4
   - From Bank Name: 80178 categorias → REMOVIDA
   - From Entity Name: 641590 categorias → REMOVIDA
   - To Bank Name: 41882 categorias → REMOVIDA
   - To Entity Name: 664021 categorias → REMOVIDA

📊 Colunas numéricas (26):
   - Amount Received
   - Amount Paid
   - txn_count_1h_velocity
   - amount_sum_1h_velocity
   - amount_mean_1h_velocity
   - amount_max_1h_velocity
   - txn_count_24h_velocity
   - amount_sum_24h_velocity
   - amount_mean_24h_velocity
   - amount_max_2

In [23]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StandardScaler, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.sql import functions as F

# 1. Pipeline para features numéricas
# Imputação para colunas numéricas
numeric_imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=[f"{col}_imputed" for col in numeric_cols],
    strategy="median"
)

# Assemblar as features numéricas imputadas em um único vetor
numeric_assembler = VectorAssembler(
    inputCols=[f"{col}_imputed" for col in numeric_cols],
    outputCol="numeric_features"
)

# Escalonar as features numéricas
numeric_scaler = StandardScaler(
    inputCol="numeric_features",
    outputCol="scaled_numeric_features"
)

# 2. Pipeline para features categóricas
categorical_stages = []
indexed_categorical_cols = []
ohe_categorical_cols = []

for col_name in categorical_cols:
    # StringIndexer para converter categorias string em índices numéricos
    indexer = StringIndexer(
        inputCol=col_name,
        outputCol=f"{col_name}_indexed",
        handleInvalid="keep" # Trata valores inválidos/nulos como uma nova categoria
    )
    categorical_stages.append(indexer)
    indexed_categorical_cols.append(f"{col_name}_indexed")

    # OneHotEncoder para converter índices numéricos em vetores one-hot
    encoder = OneHotEncoder(
        inputCols=[f"{col_name}_indexed"],
        outputCols=[f"{col_name}_encoded"],
        dropLast=False # Mantém todas as categorias, similar a sparse_output=False do sklearn
    )
    categorical_stages.append(encoder)
    ohe_categorical_cols.append(f"{col_name}_encoded")

# 3. Final Vector Assembler para combinar todas as features
# Todas as features processadas (numéricas escalonadas + categóricas one-hot encoded)
final_feature_cols = ["scaled_numeric_features"] + ohe_categorical_cols
final_assembler = VectorAssembler(
    inputCols=final_feature_cols,
    outputCol="features"
)

# Criar o pipeline completo do PySpark ML
preprocessor_spark = Pipeline(stages=[
    numeric_imputer,
    numeric_assembler,
    numeric_scaler
] + categorical_stages + [final_assembler])


print("\n" + "="*80)
print("PIPELINE CONSTRUÍDO COM PYSPARK ML")
print("="*80)

print(f"\n📦 Transformadores:")
print(f"   - Numérico: Imputer (median) + VectorAssembler + StandardScaler")
print(f"   - Categórico: StringIndexer (handleInvalid='keep') + OneHotEncoder")
print(f"   - Final: VectorAssembler para combinar todas as features")

print(f"\n✓ Pipeline pronto para fit/transform (PySpark ML)")


PIPELINE CONSTRUÍDO COM PYSPARK ML

📦 Transformadores:
   - Numérico: Imputer (median) + VectorAssembler + StandardScaler
   - Categórico: StringIndexer (handleInvalid='keep') + OneHotEncoder
   - Final: VectorAssembler para combinar todas as features

✓ Pipeline pronto para fit/transform (PySpark ML)


Treinamento do modelo

In [24]:
import pandas as pd
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("\n" + "="*80)
print("FIT E TRANSFORM (PySpark ML)")
print("="*80)

print("\n⏳ Fitting o pipeline no treino...")
model_spark = preprocessor_spark.fit(X_train)
print("✓ Fit concluído!")

print("\n⏳ Transforming treino...")
X_train_transformed_spark = model_spark.transform(X_train)

print("\n⏳ Transforming OOT...")
X_oot_transformed_spark = model_spark.transform(X_oot)
print("✓ Transform concluído!")

print("\n" + "="*80)
print("TRANSFORMAÇÃO CONCLUÍDA (PySpark ML)")
print("="*80)

print("\n📊 Schema do X_train_transformed_spark (com a nova coluna 'features'):")
X_train_transformed_spark.printSchema()

print("\n📊 Primeiras 5 linhas do X_train_transformed_spark:")
X_train_transformed_spark.show(5)

# Obter o número de features no vetor 'features'
# A metadata da coluna 'features' contém informações sobre os atributos
# num_final_features = X_train_transformed_spark.schema['features'].metadata['ml_attr']['num_attrs']
# print(f"\n📝 Número total de features no vetor 'features': {num_final_features}")

print("\n" + "="*80)
print("VERIFICAÇÕES PÓS-TRANSFORMAÇÃO (PySpark ML)")
print("="*80)

print("\n✅ Imputação e escalonamento foram aplicados pelo pipeline PySpark ML.")
print("   A coluna 'features' agora contém os dados numéricos escalonados e categóricos one-hot encoded.")

print("\n✅ A validação anti-leakage é garantida pelo processo de 'fit' exclusivo nos dados de treino e 'transform' em ambos os conjuntos.")

# --------------------------------------------------------------------------------
# 1. Prepare Training and OOT Data for PySpark ML
#    - Rename 'Is Laundering' to 'label'
#    - Join X and y dataframes (assuming row alignment)
# --------------------------------------------------------------------------------
print("="*80)
print("PREPARANDO DADOS PARA PYSPARK ML")
print("="*80)

# Add a temporary row_idx for robust joining, assuming X_train and y_train are implicitly aligned
window_spec = Window.orderBy(F.monotonically_increasing_id())

X_train_with_idx = X_train_transformed_spark.withColumn("row_idx", F.row_number().over(window_spec))
y_train_with_idx = y_train.withColumnRenamed("Is Laundering", "label").withColumn("row_idx", F.row_number().over(window_spec))

training_df = X_train_with_idx.join(y_train_with_idx, on="row_idx").drop("row_idx")

X_oot_with_idx = X_oot_transformed_spark.withColumn("row_idx", F.row_number().over(window_spec))
y_oot_with_idx = y_oot.withColumnRenamed("Is Laundering", "label").withColumn("row_idx", F.row_number().over(window_spec))

oot_df = X_oot_with_idx.join(y_oot_with_idx, on="row_idx").drop("row_idx")

print(f"\n📊 Treino (training_df):")
print(f"   Contagem de registros: {training_df.count()}")
label_1_count_train = training_df.filter(F.col('label') == 1).count()
label_0_count_train = training_df.filter(F.col('label') == 0).count()
if training_df.count() > 0:
    print(f"   Taxa de lavagem: {label_1_count_train / training_df.count() * 100:.2f}%")
else:
    print("   DataFrame de treino está vazio.")

print(f"\n📊 OOT (oot_df):")
print(f"   Contagem de registros: {oot_df.count()}")
label_1_count_oot = oot_df.filter(F.col('label') == 1).count()
label_0_count_oot = oot_df.filter(F.col('label') == 0).count()
if oot_df.count() > 0:
    print(f"   Taxa de lavagem: {label_1_count_oot / oot_df.count() * 100:.2f}%")
else:
    print("   DataFrame OOT está vazio.")

# --------------------------------------------------------------------------------
# 2. Handle Class Imbalance (Random Under Sampling)
# --------------------------------------------------------------------------------
print("\n" + "="*80)
print("BALANCEAMENTO DE CLASSES (Random Under Sampling)")
print("="*80)

positive_count = label_1_count_train
negative_count = label_0_count_train

if positive_count == 0 or negative_count == 0:
    print("⚠️  Uma das classes está vazia no conjunto de treino. Não é possível balancear.")
    balanced_training_df = training_df
else:
    # Determine sampling ratio for the majority class (negative)
    # to match the minority class (positive)
    sampling_ratio = positive_count / negative_count
    class_weights = {0: sampling_ratio, 1: 1.0} # Keep minority (1) as is, sample majority (0)
    balanced_training_df = training_df.sampleBy("label", fractions=class_weights, seed=42)

print(f"\n   training_df original count: {training_df.count()}")
print(f"   balanced_training_df count: {balanced_training_df.count()}")
if balanced_training_df.count() > 0:
    print(f"   Taxa de lavagem (balanced_training_df): {balanced_training_df.filter(F.col('label') == 1).count() / balanced_training_df.count() * 100:.2f}%")
else:
    print("   balanced_training_df está vazia após balanceamento.")


# --------------------------------------------------------------------------------
# 3. Define PySpark ML Classifiers
# --------------------------------------------------------------------------------
print("\n" + "="*80)
print("CONFIGURANDO MODELOS PYSPARK ML")
print("="*80)

spark_models = {
    'Logistic Regression': LogisticRegression(featuresCol='features', labelCol='label', elasticNetParam=0.0, regParam=0.0, maxIter=100),
    'Decision Tree': DecisionTreeClassifier(featuresCol='features', labelCol='label', maxDepth=10),
    'Random Forest': RandomForestClassifier(featuresCol='features', labelCol='label', numTrees=100, maxDepth=10),
    'GBTClassifier': GBTClassifier(featuresCol='features', labelCol='label', maxDepth=5, maxIter=100)
}

print(f"\n✅ {len(spark_models)} modelos configurados para PySpark ML.")
for model_name in spark_models.keys():
    print(f"   - {model_name}")

# --------------------------------------------------------------------------------
# 4. Train and Evaluate Models (Simplified: single OOT evaluation)
# --------------------------------------------------------------------------------
print("\n" + "="*80)
print("TREINAMENTO E AVALIAÇÃO EM PYSPARK ML (OOT)")
print("="*80)

evaluator_roc_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_pr_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")

results = {}

for model_name, classifier in spark_models.items():
    print(f"\n🔄 Treinando e avaliando {model_name}...")

    # Train model
    model = classifier.fit(balanced_training_df)

    # Make predictions on OOT data
    predictions = model.transform(oot_df)

    # Evaluate ROC AUC and PR AUC
    roc_auc = evaluator_roc_auc.evaluate(predictions)
    pr_auc = evaluator_pr_auc.evaluate(predictions)

    # Calculate other metrics manually (Precision, Recall, F1, Confusion Matrix)
    # Using default threshold 0.5 for binary classification from 'prediction' column
    tp = predictions.filter((F.col("label") == 1) & (F.col("prediction") == 1)).count()
    fp = predictions.filter((F.col("label") == 0) & (F.col("prediction") == 1)).count()
    fn = predictions.filter((F.col("label") == 1) & (F.col("prediction") == 0)).count()
    tn = predictions.filter((F.col("label") == 0) & (F.col("prediction") == 0)).count()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0.0

    results[model_name] = {
        'ROC AUC': roc_auc,
        'PR AUC': pr_auc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Accuracy': accuracy,
        'Confusion Matrix': (tn, fp, fn, tp) # Storing as tuple for easy access
    }

    print(f"\n  ✅ {model_name} - Resultados (OOT):")
    print(f"     ROC AUC: {roc_auc:.4f}")
    print(f"     PR AUC: {pr_auc:.4f}")
    print(f"     Precision: {precision:.4f}")
    print(f"     Recall (Sensitividade): {recall:.4f}")
    print(f"     F1-Score: {f1:.4f}")
    print(f"     Accuracy: {accuracy:.4f}")
    print(f"     Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

print("\n" + "="*80)
print("✅ AVALIAÇÃO DOS MODELOS CONCLUÍDA")
print("="*80)

# --------------------------------------------------------------------------------
# 5. Summarize Results
# --------------------------------------------------------------------------------
print("\n" + "="*80)
print("RANKING DE MODELOS (por PR AUC)")
print("="*80)

# Convert results to a pandas DataFrame for easier display/sorting
results_pd = pd.DataFrame(results).T
results_pd = results_pd.sort_values('PR AUC', ascending=False)
print(results_pd)

best_model_name = results_pd.index[0]
print(f"\n🏆 Melhor Modelo: {best_model_name}")
print(f"   PR AUC: {results_pd.loc[best_model_name, 'PR AUC']:.4f}")

print("\n" + "="*80)
print("NOTA SOBRE LIMITAÇÕES DA CONVERSÃO")
print("="*80)
print("\n• TimeSeriesSplit e Validação Cruzada Customizada: A implementação direta de TimeSeriesSplit do scikit-learn e validação cruzada com amostragem balanceada dentro de cada fold é complexa e ineficiente em PySpark para grandes datasets. Optamos por uma avaliação direta no conjunto OOT após balanceamento do treino.")
print("\n• Métricas KS e PSI: A implementação nativa e eficiente dessas métricas em PySpark MLlib é complexa e não trivial. Foram omitidas na conversão direta. Elas exigiriam cálculos mais avançados com UDFs ou agregação para serem obtidas em Spark.")
print("\n• Otimização de Threshold e Matriz de Custo: Similarmente, a otimização de threshold com base em custos variáveis é mais avançada e requer iterações que são mais custosas em Spark. A avaliação foi feita com o threshold padrão de 0.5.")
print("\n• Visualizações: Gráficos gerados com Matplotlib/Seaborn exigiriam a conversão de DataFrames Spark para Pandas (`.toPandas()`), o que pode ser inviável para grandes volumes de dados devido ao consumo de memória do driver. As visualizações foram omitidas.")


FIT E TRANSFORM (PySpark ML)

⏳ Fitting o pipeline no treino...
✓ Fit concluído!

⏳ Transforming treino...

⏳ Transforming OOT...
✓ Transform concluído!

TRANSFORMAÇÃO CONCLUÍDA (PySpark ML)

📊 Schema do X_train_transformed_spark (com a nova coluna 'features'):
root
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- From Bank Name: string (nullable = true)
 |-- From Entity Name: string (nullable = true)
 |-- To Bank Name: string (nullable = true)
 |-- To Entity Name: string (nullable = true)
 |-- txn_count_1h_velocity: double (nullable = true)
 |-- amount_sum_1h_velocity: double (nullable = true)
 |-- amount_mean_1h_velocity: double (nullable = true)
 |-- amount_max_1h_velocity: double (nullable = true)
 |-- txn_count_24h_velocity: double (nullable = true)
 |-- amount_sum_24h_velocity: double (nullab

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [1]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.sql.functions import col, lit, when, ceil, log
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.types import IntegerType, ArrayType, DoubleType # Added ArrayType, DoubleType

print("\n" + "="*80)
print("CÁLCULO DE KS E PSI")
print("="*80)

# --- 1. Obter o melhor modelo e gerar previsões para Treino e OOT ---
# Reconstruir o dicionário de classificadores para obter o classificador do melhor modelo
sPark_models_classifiers = {
    'Logistic Regression': LogisticRegression(featuresCol='features', labelCol='label', elasticNetParam=0.0, regParam=0.0, maxIter=100),
    'Decision Tree': DecisionTreeClassifier(featuresCol='features', labelCol='label', maxDepth=10),
    'Random Forest': RandomForestClassifier(featuresCol='features', labelCol='label', numTrees=100, maxDepth=10),
    'GBTClassifier': GBTClassifier(featuresCol='features', labelCol='label', maxDepth=5, maxIter=100)
}

# Get the best model name from the previous results_pd (assuming it's sorted by PR AUC)
# If results_pd is not available, you might need to re-run the training loop or manually set it.
# For now, let's assume `best_model_name` is still in scope from previous execution.

# If `results_pd` is not directly available or updated, let's find the best model again
# This part assumes `results` dictionary from previous execution is available
if 'results' in globals():
    results_pd_temp = pd.DataFrame(results).T
    results_pd_temp = results_pd_temp.sort_values('PR AUC', ascending=False)
    best_model_name = results_pd_temp.index[0]
else:
    print("WARNING: `results` dictionary not found. Assuming 'Decision Tree' as best model for demonstration.")
    best_model_name = 'Decision Tree' # Fallback if results are not in scope

print(f"\n🏆 Melhor modelo (baseado em PR AUC): {best_model_name}")

best_classifier = spark_models_classifiers[best_model_name]

print("\n⏳ Re-treinando o melhor modelo para obter previsões completas...")
best_model = best_classifier.fit(balanced_training_df)

print("\n⏳ Gerando previsões para o conjunto de treino (referência PSI)...")
training_predictions = best_model.transform(training_df).withColumn(
    "probability_1", col("probability").cast(ArrayType(DoubleType()))[1] # Changed this line
)

print("\n⏳ Gerando previsões para o conjunto OOT...")
oot_predictions = best_model.transform(oot_df).withColumn(
    "probability_1", col("probability").cast(ArrayType(DoubleType()))[1] # Changed this line
)
print("✓ Previsões geradas com sucesso!")

# --- 2. Cálculo do KS (Kolmogorov-Smirnov) para o conjunto OOT ---
print("\n--- Calculando KS Statistic para OOT ---")

# Criar uma coluna que rankeia as probabilidades para cálculo da CDF
window_spec_prob = Window.orderBy(col("probability_1").asc())

# Calcular CDF para a classe 0 e 1
cum_dist = oot_predictions.withColumn(
    "cum_pct_0",
    F.sum(when(col("label") == 0, lit(1)).otherwise(lit(0))).over(window_spec_prob) / oot_predictions.filter(col("label") == 0).count()
).withColumn(
    "cum_pct_1",
    F.sum(when(col("label") == 1, lit(1)).otherwise(lit(0))).over(window_spec_prob) / oot_predictions.filter(col("label") == 1).count()
)

# Calcular a diferença absoluta e encontrar o máximo
ks_statistic_row = cum_dist.select(F.max(F.abs(col("cum_pct_0") - col("cum_pct_1"))).alias("KS_Statistic")).collect()
ks_statistic = ks_statistic_row[0]["KS_Statistic"]

print(f"✅ KS Statistic (OOT): {ks_statistic:.4f}")

# --- 3. Cálculo do PSI (Population Stability Index) para OOT vs Treino ---
print("\n--- Calculando PSI (OOT vs Treino) ---")

num_bins = 10 # Número de bins para o PSI

# Calcular quantis para definir os limites dos bins no conjunto de treino
quantiles = training_predictions.approxQuantile("probability_1", [float(i)/num_bins for i in range(num_bins + 1)], 0.01)

def assign_bin(prob, quantiles_list):
    # Atribuir o bin com base nos quantis
    for i in range(len(quantiles_list) - 1):
        if quantiles_list[i] <= prob <= quantiles_list[i+1]:
            return i
    return len(quantiles_list) - 2 # last bin for max value

# Registrar a UDF
assign_bin_udf = F.udf(lambda prob: assign_bin(prob, quantiles), IntegerType())

# Atribuir bins às previsões de treino
training_binned = training_predictions.withColumn("bin", assign_bin_udf(col("probability_1")))

# Atribuir bins às previsões OOT
oot_binned = oot_predictions.withColumn("bin", assign_bin_udf(col("probability_1")))

# Contagem de cada bin para treino
training_bin_counts = training_binned.groupBy("bin").count().withColumnRenamed("count", "train_count")

# Contagem de cada bin para OOT
oot_bin_counts = oot_binned.groupBy("bin").count().withColumnRenamed("count", "oot_count")

# Juntar e calcular percentuais
psi_df = training_bin_counts.join(oot_bin_counts, on="bin", how="full_outer") \
    .fillna(0) \
    .withColumn("train_pct", col("train_count") / training_predictions.count()) \
    .withColumn("oot_pct", col("oot_count") / oot_predictions.count())

# Calcular PSI para cada bin e somar
psi_result_df = psi_df.withColumn("psi_component",
    when( (col("train_pct") == 0) | (col("oot_pct") == 0), lit(0)) # Handle cases where a bin is empty in train or OOT
    .otherwise((col("oot_pct") - col("train_pct")) * log(col("oot_pct") / col("train_pct")))
)

psi_statistic_row = psi_result_df.agg(F.sum("psi_component").alias("PSI_Statistic")).collect()
psi_statistic = psi_statistic_row[0]["PSI_Statistic"]

print(f"✅ PSI Statistic (OOT vs Treino): {psi_statistic:.4f}")

print("\n" + "="*80)
print("CÁLCULO DE KS E PSI CONCLUÍDO")
print("="*80)



CÁLCULO DE KS E PSI


AssertionError: 

In [ ]:
import shap
import pandas as pd
from sklearn.tree import DecisionTreeClassifier # Assuming Decision Tree was the best model
import matplotlib.pyplot as plt

print("\n" + "="*80)
print("EXPLICABILIDADE DO MODELO COM SHAP")
print("="*80)

# --- 1. Extrair Nomes das Features ---
# A metadata da coluna 'features' contém os nomes após o VectorAssembler
feature_names = []
feature_attrs = X_train_transformed_spark.schema["features"].metadata["ml_attr"]["attrs"]

for attr_type in feature_attrs:
    for feature_info in feature_attrs[attr_type]:
        feature_names.append(feature_info['name'])

print(f"Total de features para SHAP: {len(feature_names)}")
print("Primeiras 10 features:\n", feature_names[:10])

# --- 2. Preparar Dados para Scikit-learn (Amostragem para evitar OOM) ---
print("\n⏳ Amostrando e convertendo dados para Pandas (para SHAP)...")

# Amostrar o DataFrame Spark antes de converter para Pandas para evitar problemas de memória
# Usaremos uma amostra maior para o treino do modelo SHAP, mas uma amostra menor para calcular os shap values
# (a menos que o dataset seja pequeno o suficiente)

sample_fraction_train = 0.1 # Ex: 10% do treino para treinar o modelo sklearn
sample_fraction_oot_shap = 0.001 # Ex: 0.1% do OOT para calcular SHAP values

# Converter apenas as colunas 'features' e 'label'
training_df_pd_sample = balanced_training_df.select("features", "label").sample(False, sample_fraction_train, seed=42).toPandas()
oot_df_pd_sample = oot_df.select("features", "label").sample(False, sample_fraction_oot_shap, seed=42).toPandas()

X_train_pd = pd.DataFrame(training_df_pd_sample['features'].tolist(), columns=feature_names)
y_train_pd = training_df_pd_sample['label']

X_oot_pd_shap = pd.DataFrame(oot_df_pd_sample['features'].tolist(), columns=feature_names)
y_oot_pd_shap = oot_df_pd_sample['label']

print(f"Tamanho da amostra de treino para Sklearn: {len(X_train_pd)} linhas")
print(f"Tamanho da amostra OOT para cálculo de SHAP: {len(X_oot_pd_shap)} linhas")

# --- 3. Treinar um modelo Scikit-learn equivalente ---
print("\n⏳ Treinando Decision Tree (sklearn) na amostra de treino...")
sklearn_model = DecisionTreeClassifier(max_depth=10, random_state=42) # Usar mesmos parâmetros do melhor modelo PySpark
sklearn_model.fit(X_train_pd, y_train_pd)
print("✓ Modelo Scikit-learn treinado!")

# --- 4. Calcular SHAP Values ---
print("\n⏳ Calculando SHAP values (isso pode levar um tempo, dependendo do tamanho da amostra)...")
explainer = shap.TreeExplainer(sklearn_model)
shap_values = explainer.shap_values(X_oot_pd_shap)
print("✓ SHAP values calculados!")

# --- 5. Visualizar SHAP Results ---
print("\n--- Visualizações SHAP ---")

# Resumo global da importância das features
print("Gerando Summary Plot...")
shap.summary_plot(shap_values[1], X_oot_pd_shap, feature_names=feature_names, show=False)
plt.title("SHAP Summary Plot (Classe Positiva)")
plt.show()

# Plot para uma única instância (opcional)
# if len(X_oot_pd_shap) > 0:
#     print("Gerando Force Plot para a primeira instância...")
#     shap.initjs()
#     display(shap.force_plot(explainer.expected_value[1], shap_values[1][0,:], X_oot_pd_shap.iloc[0,:], feature_names=feature_names))
# else:
#     print("Não há instâncias suficientes na amostra OOT para gerar Force Plot.")

print("\n" + "="*80)
print("EXPLICABILIDADE SHAP CONCLUÍDA")
print("="*80)
